#Recommendation System

### 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler

### 2. Load Dataset

In [ ]:
df = pd.read_csv("anime.csv")

print(df.head())
print(df.info())


   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  

### 3. Data Preprocessing

In [ ]:
#Handling the missing values
df = df.dropna(subset=['name', 'genre', 'rating', 'members'])

In [ ]:
# Reset index
df = df.reset_index(drop=True)

### 4. Feature Extraction

In [ ]:
# Convert genre string to list
df['genre'] = df['genre'].apply(lambda x: x.split(', '))


In [ ]:
# One-hot encode genres
mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(df['genre'])

In [ ]:
# Convert to DataFrame
genre_df = pd.DataFrame(genre_encoded, columns=mlb.classes_)

In [ ]:
# Normalize numerical features
scaler = MinMaxScaler()
num_features = scaler.fit_transform(df[['rating', 'members']])
num_df = pd.DataFrame(num_features, columns=['rating', 'members'])

In [ ]:
# Combine features
features = pd.concat([genre_df, num_df], axis=1)

###  5. Cosine Similarity

In [ ]:
cos_sim = cosine_similarity(features)

In [ ]:
# Check similarity range
print("Min similarity:", cos_sim.min())
print("Max similarity:", cos_sim.max())

Min similarity: 0.0
Max similarity: 1.0000000000000004


#### 6. Recommendation Function

In [ ]:
def recommend_anime_threshold(anime_name, threshold=0.5, top_n=5):

    # Check if anime exists
    if anime_name not in df['name'].values:
        return "Anime not found!"

    idx = df[df['name'] == anime_name].index[0]

    sim_scores = list(enumerate(cos_sim[idx]))

    # Filter using threshold
    sim_scores = [i for i in sim_scores if i[1] >= threshold and i[0] != idx]

    # Sort
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Take top_n
    sim_scores = sim_scores[:top_n]

    anime_indices = [i[0] for i in sim_scores]

    return df['name'].iloc[anime_indices]


### 7. Test Recommendation

In [ ]:
def recommend_anime(anime_name, top_n=5):

    # Check if anime exists
    if anime_name not in df['name'].values:
        return "Anime not found!"

    # Find index
    idx = df[df['name'] == anime_name].index[0]

    # Similarity scores
    sim_scores = list(enumerate(cos_sim[idx]))

    # Sort
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Top N (excluding itself)
    sim_scores = sim_scores[1:top_n+1]

    anime_indices = [i[0] for i in sim_scores]

    return df['name'].iloc[anime_indices]


In [ ]:
print("Recommendations:")
print(recommend_anime("Naruto", top_n=5))

Recommendations:
615                                    Naruto: Shippuuden
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
1573    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
Name: name, dtype: object


In [ ]:
print("\nThreshold = 0.3")
print(recommend_anime_threshold("Naruto", threshold=0.3))

print("\nThreshold = 0.5")
print(recommend_anime_threshold("Naruto", threshold=0.5))

print("\nThreshold = 0.7")
print(recommend_anime_threshold("Naruto", threshold=0.7))


Threshold = 0.3
615                                    Naruto: Shippuuden
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
1573    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
Name: name, dtype: object

Threshold = 0.5
615                                    Naruto: Shippuuden
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
1573    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
Name: name, dtype: object

Threshold = 0.7
615                                    Naruto: Shippuuden
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
1573    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
Name: name, dtype: object


# Observations
##### Lower threshold (0.3) → More recommendations but less similarity
##### Medium threshold (0.5) → Balanced recommendations
##### High threshold (0.7) → Fewer but highly similar anime


##Similarity Threshold Explanation
#####Similarity score ranges from 0 to 1
#####Higher value → more similar items
#####Threshold helps filter quality vs quantity

##Example:

#####0.3 → broad recommendations
#####0.7 → very strict recommendations

##Advantages of Content-Based Recommendation
#####Works without user history
#####Personalized recommendations
#####No cold-start for items
##Limitations
#####Limited diversity
#####Depends on feature quality
#####Cannot discover totally new types

**Interview Questions**
##Difference: User-Based vs Item-Based Collaborative Filtering


🔹 User-Based
Finds similar users
Recommends what similar users liked

👉 Example:
"If user A likes Naruto, and user B is similar → recommend B's favorites"

🔹 Item-Based
Finds similar items (anime)
Recommends similar anime

👉 Example:
"If Naruto is similar to One Piece → recommend One Piece"

## What is Collaborative Filtering?

👉 Technique used in recommendation systems

🔹 How it works:
Uses user behavior (ratings, likes)
Finds patterns
Recommends items based on similarity
🔹 Types:
User-based
Item-based

##Types:
######User-based
######Item-based

# Conclusion

##### A content-based recommendation system was successfully implemented using cosine similarity.
##### Genre and numerical features (rating, members) were used to compute similarity.
##### Threshold tuning helped control recommendation quality.
##### Lower threshold gives more results, higher threshold gives more accurate results.
##### The system effectively recommends similar anime based on content features.
